In [1]:
import sys

sys.path.append("../..")

In [16]:
# import os
# os.environ['PROJ_LIB'] = '/opt/anaconda3/envs/claymodel/share/proj'

# explicitly set the PROJ data directory to ensure pyproj can resolve EPSG codes correctly.
from pyproj.datadir import set_data_dir

set_data_dir("/opt/anaconda3/envs/claymodel/share/proj")

In [17]:
import numpy as np
import pandas as pd
import pystac_client
import stackstac
from rasterio.enums import Resampling
import odc.stac
import planetary_computer
import matplotlib.pyplot as plt
from pystac.extensions.eo import EOExtension as eo

In [15]:
CLAY_TO_S2 = {
    "blue":      "B02",
    "green":     "B03",
    "red":       "B04",
    "rededge1":  "B05",
    "rededge2":  "B06",
    "rededge3":  "B07",
    "nir":       "B08",
    "nir08":     "B8A",
    "swir16":    "B11",
    "swir22":    "B12",
}


In [18]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [19]:
#bbox and time


bbox = [77.95, 30.20, 78.20, 30.45]

time = "2024-01-01/2026-01-15"

In [20]:
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time,
    query={
        "eo:cloud_cover": {"lt": 20}
    }
)

items = list(search.get_items())
print(f"Scenes found: {len(items)}")


/opt/anaconda3/envs/odcenv/lib/python3.11/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


Scenes found: 166


In [21]:
item = items[0]

print(item.datetime)
print(item.properties.get("proj:epsg"))
print(item.bbox)


2026-01-08 05:31:09.024000+00:00
None
[77.8679825, 29.7058729, 79.0187982, 30.7178664]


In [22]:
assets = list(CLAY_TO_S2.values())

In [23]:
assets

['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']

In [24]:
cube = stackstac.stack(
    items,
    assets=assets,
    epsg=4326,
    resolution=10,
    bounds_latlon=bbox,
    chunksize=2048)